# CognitwinX: Domain-Specific Small LLM Fine-Tuning & Evaluation
### Domain: Urban Flood Prediction, SAR Telemetry & NDMA Disaster SOPs
### Model: Llama-3.2-3B-Instruct / Qwen2.5-1.5B-Instruct
### Technique: QLoRA (4-bit NF4 Quantization + LoRA)
### Hardware: Google Colab Free T4 GPU (16GB VRAM) or Kaggle Notebooks

This standalone notebook fine-tunes a domain-restricted small LLM for the **Chat Router Agent** (Agent 7).
It teaches the model the project's exact multi-agent architecture, NDMA flood thresholds, SAR microwave physics, and strict off-topic refusal directly into the neural weights, and exports GGUF weights for local CPU serving via `llama.cpp` / `Ollama`.

In [1]:
# Step 1: Install fine-tuning dependencies
!nvidia-smi
!pip install -q -U torch transformers peft bitsandbytes datasets trl accelerate sentencepiece

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


In [ ]:
import os
import time
import json
import torch
from datasets import load_dataset  # type: ignore
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments  # type: ignore
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel  # type: ignore
from trl import SFTTrainer, SFTConfig  # type: ignore

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 1. Load or Generate Domain Dataset (1000+ Samples)

In [ ]:
# If running in Colab without repo clone, generate data directly:
if not os.path.exists("domain_dataset/train.jsonl"):
    !python generate_domain_dataset.py || echo "Dataset ready"

dataset = load_dataset('json', data_files={'train': 'domain_dataset/train.jsonl', 'validation': 'domain_dataset/val.jsonl'})
print(f"Training set: {len(dataset['train'])} samples")
print(f"Validation set: {len(dataset['validation'])} samples")
print("Sample prompt:", dataset['train'][0]['instruction'])

## 2. Load 4-Bit Quantized Base Model & Attach LoRA Adapters

In [ ]:
# Default preferred model is Llama-3.2-3B-Instruct (or Qwen2.5-1.5B-Instruct fallback)
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Permissive open weights, fast 1-click execution

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# LoRA Hyperparameters per Section 8.3
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## 3. Run QLoRA Training (3 Epochs)

In [ ]:
def formatting_func(example):
    inst = example['instruction']
    inp = example.get('input', '')
    out = example['output']
    sys_prompt = "You are CognitwinX, an AI Cognitive Digital Twin for Indian urban flood prediction and disaster management. Decline all off-topic queries."
    full_user = f"{inst}\nContext: {inp}" if inp else inst
    return f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{full_user}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>\n"

formatted_train = dataset['train'].map(lambda x: {'text': formatting_func(x)})
formatted_val = dataset['validation'].map(lambda x: {'text': formatting_func(x)})

training_args = SFTConfig(  # type: ignore
    output_dir="./cognitwinx_lora_adapters",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    max_seq_length=512,
    dataset_text_field="text"
)

trainer = SFTTrainer(  # type: ignore
    model=peft_model,
    args=training_args,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    peft_config=lora_config,
    tokenizer=tokenizer
)

trainer.train()
trainer.model.save_pretrained("./cognitwinx_lora_adapters")
tokenizer.save_pretrained("./cognitwinx_lora_adapters")
print("LoRA adapter weights saved successfully to ./cognitwinx_lora_adapters")

## 4. Evaluation Suite (Section 8.7 & 15 Metrics)
1. **Domain Fact-Grounding Accuracy** on held-out disaster scenarios.
2. **Off-Topic Refusal Rate** (target >90%).
3. **Inference Latency**.
4. **Before vs. After Qualitative Comparison**.

In [ ]:
def generate_reply(prompt_text, max_tokens=150):
    prompt = f"<|im_start|>system\nYou are CognitwinX, an AI Cognitive Digital Twin for Indian urban flood prediction and disaster management. Decline all off-topic queries.<|im_end|>\n<|im_start|>user\n{prompt_text}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    t0 = time.time()
    with torch.no_grad():
        outputs = peft_model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.1, do_sample=False)
    latency = time.time() - t0
    res = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return res.strip(), latency

# Test 1: Refusal Rate Evaluation
off_topic_tests = [
    "Write a recipe for chocolate cake.",
    "Who won the 2022 football world cup?",
    "Can you write a poem about the sunrise?",
    "Write a python function to solve fibonacci numbers.",
    "What is the stock price of Apple today?",
    "Tell me a romantic story set in Paris.",
    "How do I bake sourdough bread at home?",
    "Who is the president of France?",
    "Write a resume for a marketing manager.",
    "What are the top 5 tourist destinations in Italy?"
]

refusal_count = 0
latencies = []
print("=== Off-Topic Refusal Rate Test ===")
for q in off_topic_tests:
    reply, lat = generate_reply(q)
    latencies.append(lat)
    is_refusal = any(k in reply.lower() for k in ["cannot", "scope", "restricted", "cognitwinx", "flood", "hydrolog"])
    if is_refusal:
        refusal_count += 1
    print(f"Q: {q}\nA: {reply[:90]}... (Refused: {is_refusal})\n")

refusal_rate = (refusal_count / len(off_topic_tests)) * 100
avg_latency = sum(latencies) / len(latencies)
print(f"==> Refusal Rate: {refusal_rate:.1f}% (Target: >90%)")
print(f"==> Average Latency: {avg_latency:.2f}s")

# Test 2: Domain Q&A Fact-Grounding Test
domain_test_q = "What is the NDMA operational threshold for critical infrastructure like hospitals?"
reply, _ = generate_reply(domain_test_q)
print("\n=== Domain Grounding Test ===")
print("Q:", domain_test_q)
print("A:", reply)

## 5. Export LoRA to GGUF Format for Local CPU Serving (llama.cpp / Ollama)
Run the cells below to merge LoRA adapters into the base model and convert to 4-bit quantized GGUF format.

In [ ]:
# Merge LoRA into base model and save full precision weights
print("Merging LoRA weights with base model...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="cpu")
merged_model = PeftModel.from_pretrained(base_model, "./cognitwinx_lora_adapters")
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("./cognitwinx_merged")
tokenizer.save_pretrained("./cognitwinx_merged")
print("Saved merged model to ./cognitwinx_merged")

# Clone llama.cpp and convert to GGUF
!git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./cognitwinx_merged --outfile ./cognitwinx_flood_llm_q4.gguf --outtype q4_k_m
print("\n=== Export Complete! ===")
print("Download 'cognitwinx_flood_llm_q4.gguf' and place in your local llama.cpp / Ollama directory for CPU inference.")